# HBAAC 2026: End-to-End Pipeline

Notebook này chứa toàn bộ đường ống xử lý dữ liệu từ đầu đến cuối, bao gồm:
1. Tự động tải dữ liệu từ Google Drive qua gdown
2. Làm sạch dữ liệu chặt chẽ (Data Cleaning)
3. Tạo lưới thời gian liên tục và Kỹ nghệ Đặc trưng
4. Huấn luyện mô hình LightGBM chặn sớm
5. Suy luận và xuất kết quả định dạng chuẩn cuộc thi

## 1. Khởi Tạo Thư Viện

In [ ]:
import os
import gc
import logging
from pathlib import Path
from typing import Dict, Tuple
import numpy as np
import pandas as pd
import lightgbm as lgb
import gdown
import warnings
from sklearn.metrics import mean_squared_error
warnings.filterwarnings('ignore')

## 2. Tự Động Tải Dữ Liệu
Khởi tạo cấu trúc thư mục làm việc cục bộ và nạp dữ liệu từ kho lưu trữ đám mây chia sẻ công khai.

In [2]:
os.makedirs('data/raw', exist_ok=True)
os.makedirs('submissions', exist_ok=True)

train_id = 'Dien_ID_File_Train_Cua_Ban_Vao_Day'
sample_id = 'Dien_ID_File_Sample_Submission_Cua_Ban_Vao_Day'

gdown.download(f'https://drive.google.com/uc?id=1NWxS9Z-U54ib0xDP1VefHtam1GRbYbA_', 'data/raw/train.csv', quiet=False)
gdown.download(f'https://drive.google.com/uc?id=1JG-Z2RowstWQCgEbI4VFR45pxmlc3U5k', 'data/raw/sample_submission.csv', quiet=False)

DEFAULT_TRAIN_PATH = "data/raw/train.csv"
DEFAULT_SAMPLE_PATH = "data/raw/sample_submission.csv"

Downloading...
From: https://drive.google.com/uc?id=1NWxS9Z-U54ib0xDP1VefHtam1GRbYbA_
To: d:\Study source\HBAAC\HBAAC---Last-Day\notebooks\data\raw\train.csv
100%|██████████| 45.8M/45.8M [00:04<00:00, 9.94MB/s]
Downloading...
From: https://drive.google.com/uc?id=1JG-Z2RowstWQCgEbI4VFR45pxmlc3U5k
To: d:\Study source\HBAAC\HBAAC---Last-Day\notebooks\data\raw\sample_submission.csv
100%|██████████| 2.49M/2.49M [00:01<00:00, 2.25MB/s]


## 3. Kiến Trúc Mô-đun Làm Sạch Dữ Liệu
Định nghĩa các hàm xử lý dữ liệu thô, loại bỏ mâu thuẫn hệ thống ERP, tối ưu hóa bộ nhớ an toàn và phòng chống lỗi sập ngầm.

In [3]:
logger = logging.getLogger(__name__)
SAMPLE_ID_COLUMN = "id"
SAMPLE_ID_SUFFIXES = ("_validation", "_evaluation")

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace(" ", "_", regex=False)
    )
    return df

def parse_decimal_comma(series: pd.Series) -> pd.Series:
    cleaned = (
        series.astype(str)
        .str.strip()
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False)
    )
    cleaned = cleaned.replace({
        "": np.nan,
        "nan": np.nan,
        "None": np.nan,
    })
    return pd.to_numeric(cleaned, errors="coerce")

def extract_sample_skus(sample: pd.DataFrame) -> pd.Series:
    if SAMPLE_ID_COLUMN not in sample.columns:
        raise ValueError(f"Missing column: {SAMPLE_ID_COLUMN}")
    skus = sample[SAMPLE_ID_COLUMN].astype(str).str.strip()
    for suffix in SAMPLE_ID_SUFFIXES:
        skus = skus.str.replace(suffix, "", regex=False)
    result = skus.drop_duplicates().reset_index(drop=True)
    if result.empty:
        logger.warning("No SKUs extracted from sample submission. Check sample format.")
    return result

def reduce_mem_usage(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    start_mem = df.memory_usage(deep=True).sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        if pd.api.types.is_datetime64_any_dtype(col_type):
            continue
        if pd.api.types.is_integer_dtype(col_type):
            c_min = df[col].min()
            c_max = df[col].max()
            if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.int64)
        elif pd.api.types.is_float_dtype(col_type):
            if col in ["UnitPrice", "SalesAmount", "Unit_Cost", "Cost_Amount"]:
                continue
            c_min = df[col].min()
            c_max = df[col].max()
            if (
                np.isfinite(c_min)
                and np.isfinite(c_max)
                and c_min >= np.finfo(np.float32).min
                and c_max <= np.finfo(np.float32).max
            ):
                df[col] = df[col].astype(np.float32)
            else:
                df[col] = df[col].astype(np.float64)
    end_mem = df.memory_usage(deep=True).sum() / 1024**2
    logger.info(f"Memory reduced: {start_mem:.2f} MB -> {end_mem:.2f} MB")
    return df

def read_and_clean_transactions(
    train_path: str = DEFAULT_TRAIN_PATH,
    sample_path: str = DEFAULT_SAMPLE_PATH,
    drop_exact_duplicates: bool = False,
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict]:
    raw = pd.read_csv(train_path, dtype=str)
    sample = pd.read_csv(sample_path, dtype=str)
    initial_rows = int(len(raw))
    raw = normalize_columns(raw)
    required_cols = [
        "Date",
        "Stt",
        "ItemCode",
        "Quantity",
        "UnitPrice",
        "SalesAmount",
        "Unit_Cost",
        "Cost_Amount",
    ]
    missing_cols = [
        c for c in required_cols
        if c not in raw.columns
    ]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")
    raw["Date"] = pd.to_datetime(raw["Date"], errors="coerce")
    raw["Stt"] = raw["Stt"].astype(str).str.strip()
    raw["ItemCode"] = raw["ItemCode"].astype(str).str.strip()
    numeric_cols = [
        "Quantity",
        "UnitPrice",
        "SalesAmount",
        "Unit_Cost",
        "Cost_Amount",
    ]
    for col in numeric_cols:
        raw[col] = parse_decimal_comma(raw[col])
    duplicate_subset = [
        "Date",
        "ItemCode",
        "Quantity",
        "SalesAmount",
        "Cost_Amount",
    ]
    raw["is_exact_duplicate" ] = raw.duplicated(subset=duplicate_subset, keep="first").astype(np.int8)
    total_exact_duplicates = int(raw["is_exact_duplicate"].sum())
    if drop_exact_duplicates:
        raw = raw.loc[raw["is_exact_duplicate"].eq(0)].copy()
    raw["bad_date"] = raw["Date"].isna().astype(np.int8)
    raw["bad_item_code"] = (
        raw["ItemCode"].isna()
        | raw["ItemCode"].eq("")
        | raw["ItemCode"].str.lower().eq("nan")
    ).astype(np.int8)
    for col in numeric_cols:
        raw[f"bad_{col}"] = raw[col].isna().astype(np.int8)
    critical_bad = (
        raw["bad_date"].eq(1)
        | raw["bad_item_code"].eq(1)
        | raw["bad_Quantity"].eq(1)
        | raw["bad_UnitPrice"].eq(1)
        | raw["bad_SalesAmount"].eq(1)
        | raw["bad_Cost_Amount"].eq(1)
        | raw["bad_Unit_Cost"].eq(1)
    )
    clean = raw.loc[~critical_bad].copy()
    clean["is_zero_quantity"] = (clean["Quantity"] == 0).astype(np.int8)
    clean["is_return"] = (clean["Quantity"] < 0).astype(np.int8)
    clean["is_sign_mismatch" ] = (
        (
            (clean["Quantity"] < 0)
            & (
                (clean["SalesAmount"] > 0)
                | (clean["Cost_Amount"] > 0)
            )
        )
        | (
            (clean["Quantity"] > 0)
            & (
                (clean["SalesAmount"] < 0)
                | (clean["Cost_Amount"] < 0)
            )
        )
    ).astype(np.int8)
    tolerance = 0.01
    with np.errstate(divide="ignore", invalid="ignore"):
        implied = np.where(
            clean["Quantity"] != 0,
            clean["SalesAmount"] / clean["Quantity"],
            np.nan,
        )
    clean["is_price_mismatch"] = (
        pd.Series(implied, index=clean.index).sub(clean["UnitPrice"]).abs() > tolerance
    ).astype(np.int8)
    sample_skus = set(extract_sample_skus(sample))
    clean["sku_in_sample"] = clean["ItemCode"].isin(sample_skus).astype(np.int8)
    report = {
        "initial_rows": initial_rows,
        "rows_after_cleaning": int(len(clean)),
        "clean_ratio": float(len(clean) / max(initial_rows, 1)),
        "unique_skus": int(clean["ItemCode"].nunique()),
        "date_min": str(clean["Date"].min().date()) if not clean.empty else None,
        "date_max": str(clean["Date"].max().date()) if not clean.empty else None,
        "exact_duplicate_rows": total_exact_duplicates,
        "zero_quantity_rows": int(clean["is_zero_quantity"].sum()),
        "return_rows": int(clean["is_return"].sum()),
        "sign_mismatch_rows": int(clean["is_sign_mismatch"].sum()),
        "price_mismatch_rows": int(clean["is_price_mismatch"].sum()),
    }
    clean = reduce_mem_usage(clean)
    gc.collect()
    return clean, sample, report

## 4. Thực Thi Đường Ống Làm Sạch Dữ Liệu

In [4]:
clean_df, sample_sub, report = read_and_clean_transactions(
    train_path=DEFAULT_TRAIN_PATH,
    sample_path=DEFAULT_SAMPLE_PATH,
    drop_exact_duplicates=True
)
print(report)

{'initial_rows': 711980, 'rows_after_cleaning': 673690, 'clean_ratio': 0.9462203994494227, 'unique_skus': 15972, 'date_min': '2020-11-17', 'date_max': '2025-09-05', 'exact_duplicate_rows': 38290, 'zero_quantity_rows': 3044, 'return_rows': 37100, 'sign_mismatch_rows': 31, 'price_mismatch_rows': 289892}


## 5. Xây Dựng Lưới Lịch Liên Tục & Kỹ Nghệ Đặc Trưng
Gộp nhóm dữ liệu lượng bán thực tế sang lưới thời gian liên tục cấp độ ngày để bảo toàn ý nghĩa khoảng cách vật lý khi tạo các biến trễ (Lag) và biến động trượt.

In [ ]:
"""
Feature Engineering — Vietnamese Auto Parts Demand Forecasting
Data: 2020-11-17 → 2025-09-05  |  15,972 SKUs  |  711,980 transactions
Forecast: 56 days (F1..F28 validation + F29..F56 evaluation)
Metric: WRMSSE (weighted by SKU profit)
"""

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger(__name__)

TARGET_HORIZON = 56          # total forecast horizon
VAL_HORIZON    = 28          # F1..F28  → 2025-09-06 to 2025-10-03
EVAL_HORIZON   = 28          # F29..F56 → 2025-10-04 to 2025-10-31

DATA_START = pd.Timestamp('2020-11-17')
DATA_END   = pd.Timestamp('2025-09-05')
VAL_START  = pd.Timestamp('2025-09-06')
EVAL_END   = pd.Timestamp('2025-10-31')

LAGS         = [56, 63, 70, 84]
ROLL_WINDOWS = [7, 14, 28, 56]

def parse_vnd(series: pd.Series) -> pd.Series:
    """Parse VND amounts written with comma as decimal separator."""
    return pd.to_numeric(
        series.astype(str)
              .str.replace(',', '.', regex=False)
              .str.replace(' ', '', regex=False),
        errors='coerce'
    )

def vectorized_days_to_event(
    date_series: pd.Series,
    event_dates,
    direction: str = 'next',
    clip_val: int = 45,
) -> np.ndarray:
    dates_np  = date_series.values.astype('datetime64[D]')
    events_np = np.array(pd.to_datetime(event_dates), dtype='datetime64[D]')
    result    = np.full(len(dates_np), clip_val + 1, dtype=np.int16)
    for event in events_np:
        diff = ((event - dates_np).astype(np.int16) if direction == 'next'
                else (dates_np - event).astype(np.int16))
        valid  = (diff >= 0) & (diff < result)
        result = np.where(valid, diff, result)
    return np.clip(result, 0, clip_val).astype(np.int16)

logger.info("Loading raw transactions …")

raw = pd.read_csv('data/raw/train.csv', dtype=str)
sample = pd.read_csv('data/raw/sample_submission.csv', dtype=str)

# Lấy danh sách toàn bộ SKU từ file sample submission để đảm bảo không rớt SKU
all_skus = set(sample['id'].str.split('_').str[0].unique())

raw.columns = raw.columns.str.strip().str.replace(' ', '_')
raw['Date']       = pd.to_datetime(raw['Date'], errors='coerce')
raw['Quantity']   = pd.to_numeric(raw['Quantity'], errors='coerce')
raw['SalesAmount'] = parse_vnd(raw['SalesAmount'])
raw['Cost_Amount'] = parse_vnd(raw['Cost_Amount'])
raw['profit_line'] = raw['SalesAmount'] - raw['Cost_Amount']

raw = raw.dropna(subset=['Date', 'ItemCode', 'Quantity'])
raw['ItemCode'] = raw['ItemCode'].str.strip()

logger.info("Computing profit weights …")

sku_profit = (
    raw.groupby('ItemCode')['profit_line']
       .sum()
       .clip(lower=0)
)
total_profit = sku_profit.sum()
sku_weight   = (sku_profit / total_profit).rename('profit_weight')

zero_weight_skus = set(sku_weight[sku_weight == 0].index)
logger.info(f"Zero-weight SKUs (predict 0): {len(zero_weight_skus)}")

logger.info("Building net daily sales …")

daily_net = (
    raw.groupby(['Date', 'ItemCode'], as_index=False)['Quantity']
       .sum()
)
daily_net['Quantity'] = daily_net['Quantity'].clip(lower=0).astype(np.float32)

never_sold_skus = set(
    daily_net.groupby('ItemCode')['Quantity']
             .max()
             .loc[lambda x: x == 0]
             .index
)
logger.info(f"Never-sold SKUs (predict 0): {len(never_sold_skus)}")

predict_zero_skus = zero_weight_skus | never_sold_skus
logger.info(f"Total predict-zero SKUs: {len(predict_zero_skus)}")

train_skus = all_skus - predict_zero_skus
logger.info(f"SKUs entering training pipeline: {len(train_skus)}")

daily_train = daily_net[daily_net['ItemCode'].isin(train_skus)].copy()

logger.info("Building per-SKU date grid …")

sku_bounds = (
    daily_train.groupby('ItemCode')['Date']
    .agg(['min', 'max'])
)

grid_list = []
for sku in train_skus:
    if sku in sku_bounds.index:
        row = sku_bounds.loc[sku]
    else:
        row = pd.Series({'min': np.nan, 'max': np.nan})

    if pd.isna(row['min']) or pd.isna(row['max']):
        # giữ SKU nhưng chỉ tạo 1 dòng giả (KHÔNG nhân full range)
        dates = pd.date_range(DATA_START, periods=1, freq='D')
    else:
        # expand đúng lịch sử + đảm bảo coverage đến ngày cuối
        dates = pd.date_range(start=row['min'], end=DATA_END, freq='D')
        
    grid_list.append(pd.DataFrame({
        'Date':     dates,
        'ItemCode': sku,
    }))

df = pd.concat(grid_list, ignore_index=True)
logger.info(f"Grid shape: {df.shape}")

df = df.merge(daily_train, on=['Date', 'ItemCode'], how='left')
df['Quantity'] = df['Quantity'].fillna(0).astype(np.float32)
df = df.sort_values(['ItemCode', 'Date']).reset_index(drop=True)

pandemic_dates = pd.date_range(start='2020-11-17', end='2022-12-31')
df['is_pandemic_era'] = df['Date'].isin(pandemic_dates).astype(np.int8)

holidays_vn = [
    '2021-01-01', '2022-01-01', '2023-01-01', '2024-01-01', '2025-01-01',
    '2021-02-10', '2021-02-11', '2021-02-12', '2021-02-13', '2021-02-14',
    '2022-01-31', '2022-02-01', '2022-02-02', '2022-02-03', '2022-02-04',
    '2023-01-22', '2023-01-23', '2023-01-24', '2023-01-25', '2023-01-26',
    '2024-02-08', '2024-02-09', '2024-02-10', '2024-02-11', '2024-02-12',
    '2025-01-27', '2025-01-28', '2025-01-29', '2025-01-30', '2025-01-31',
    '2021-04-21', '2022-04-10', '2023-04-29', '2024-04-18', '2025-04-07',
    '2021-04-30', '2021-05-01', '2022-04-30', '2022-05-01',
    '2023-04-30', '2023-05-01', '2024-04-30', '2024-05-01',
    '2025-04-30', '2025-05-01',
    '2021-09-02', '2022-09-02', '2023-09-02', '2024-09-02', '2025-09-02',
    '2021-11-20', '2022-11-20', '2023-11-20', '2024-11-20', '2025-11-20',
    '2021-09-21', '2022-09-10', '2023-09-29', '2024-09-17', '2025-10-06',
]
holiday_dates = pd.to_datetime(holidays_vn)

df['is_holiday'] = df['Date'].isin(holiday_dates).astype(np.int8)

df['is_giotohungvuong'] = df['Date'].isin(pd.to_datetime([
    '2021-04-21', '2022-04-10', '2023-04-29', '2024-04-18', '2025-04-07',
])).astype(np.int8)

df['is_laodong'] = df['Date'].isin(pd.to_datetime([
    '2021-04-30', '2021-05-01', '2022-04-30', '2022-05-01',
    '2023-04-30', '2023-05-01', '2024-04-30', '2024-05-01',
    '2025-04-30', '2025-05-01',
])).astype(np.int8)

df['is_nhagiao'] = df['Date'].isin(pd.to_datetime([
    '2021-11-20', '2022-11-20', '2023-11-20', '2024-11-20', '2025-11-20',
])).astype(np.int8)

df['is_trungthu'] = df['Date'].isin(pd.to_datetime([
    '2021-09-21', '2022-09-10', '2023-09-29', '2024-09-17', '2025-10-06',
])).astype(np.int8)

unique_dates = pd.DataFrame({'Date': df['Date'].sort_values().unique()})

next_hol = pd.merge_asof(
    unique_dates, pd.DataFrame({'holiday_date': holiday_dates}).sort_values('holiday_date'),
    left_on='Date', right_on='holiday_date', direction='forward',
)
next_hol['days_to_next_holiday'] = (
    (next_hol['holiday_date'] - next_hol['Date']).dt.days
    .fillna(365).clip(0, 45).astype(np.int16)
)

prev_hol = pd.merge_asof(
    unique_dates, pd.DataFrame({'holiday_date': holiday_dates}).sort_values('holiday_date'),
    left_on='Date', right_on='holiday_date', direction='backward',
)
prev_hol['days_since_last_holiday'] = (
    (prev_hol['Date'] - prev_hol['holiday_date']).dt.days
    .fillna(365).clip(0, 45).astype(np.int16)
)

df = (df
      .merge(next_hol[['Date', 'days_to_next_holiday']], on='Date', how='left')
      .merge(prev_hol[['Date', 'days_since_last_holiday']], on='Date', how='left'))

tet_windows = [
    ('2021-02-10', '2021-02-16'),
    ('2022-01-31', '2022-02-05'),
    ('2023-01-22', '2023-01-28'),
    ('2024-02-08', '2024-02-14'),
    ('2025-01-27', '2025-02-02'),
]
tet_starts = pd.to_datetime([w[0] for w in tet_windows])
tet_ends   = pd.to_datetime([w[1] for w in tet_windows])

df['is_tet'] = np.int8(0)
for s, e in tet_windows:
    df.loc[(df['Date'] >= s) & (df['Date'] <= e), 'is_tet'] = np.int8(1)

tet_start_df = pd.DataFrame({'tet_start': tet_starts}).sort_values('tet_start')
tet_merge = pd.merge_asof(
    unique_dates, tet_start_df,
    left_on='Date', right_on='tet_start', direction='forward',
)
tet_merge['days_to_tet'] = (
    (tet_merge['tet_start'] - tet_merge['Date']).dt.days
    .fillna(365).clip(0, 60).astype(np.int16)
)

tet_end_df = pd.DataFrame({'tet_end': tet_ends}).sort_values('tet_end')
post_merge = pd.merge_asof(
    unique_dates, tet_end_df,
    left_on='Date', right_on='tet_end', direction='backward',
)
post_merge['days_after_tet'] = (
    (post_merge['Date'] - post_merge['tet_end']).dt.days
    .fillna(365).clip(0, 30).astype(np.int16)
)
post_merge.loc[post_merge['days_after_tet'] == 0, 'days_after_tet'] = np.int16(365)

df = (df
      .merge(tet_merge[['Date', 'days_to_tet']], on='Date', how='left')
      .merge(post_merge[['Date', 'days_after_tet']], on='Date', how='left'))

df['pre_tet_window'] = (
    (df['days_to_tet'] <= 30) & (df['is_tet'] == 0)
).astype(np.int8)

df['post_tet_window'] = (
    (df['days_after_tet'] <= 15) & (df['days_after_tet'] < 365)
).astype(np.int8)

df['dow']   = df['Date'].dt.dayofweek.astype(np.int8) 
df['month'] = df['Date'].dt.month.astype(np.int8)
df['week']  = df['Date'].dt.isocalendar().week.astype(np.int16)

df['dow_sin']   = np.sin(2 * np.pi * df['dow']   / 7 ).astype(np.float32)
df['dow_cos']   = np.cos(2 * np.pi * df['dow']   / 7 ).astype(np.float32)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12).astype(np.float32)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12).astype(np.float32)

df['is_saturday'] = (df['dow'] == 5).astype(np.int8)

df['is_quarter_end_month'] = df['month'].isin([3, 6, 9, 12]).astype(np.int8)

logger.info("Computing SKU static stats …")

train_cutoff = DATA_END - pd.Timedelta(days=TARGET_HORIZON)

hist_sales = daily_train[daily_train['Date'] <= train_cutoff]

sku_stats = (
    hist_sales
    .groupby('ItemCode')['Quantity']
    .agg(
        sku_mean='mean',
        sku_std='std',
        sku_median=lambda x: x.median(),
        sku_max='max',
        sku_nonzero_rate=lambda x: (x > 0).mean(),
    )
    .reset_index()
)

df = df.merge(sku_stats, on='ItemCode', how='left')

df = df.merge(sku_weight.reset_index(), on='ItemCode', how='left')
df['profit_weight'] = df['profit_weight'].fillna(0).astype(np.float32)

logger.info("Computing lag features …")

g = df.groupby('ItemCode')['Quantity']

for lag in LAGS:
    df[f'lag_{lag}'] = g.shift(lag).astype(np.float32)

logger.info("Computing rolling features …")

for w in ROLL_WINDOWS:
    df[f'roll_mean_{w}'] = (
        g.transform(lambda x: x.shift(TARGET_HORIZON).rolling(w, min_periods=1).mean())
         .astype(np.float32)
    )
    df[f'roll_std_{w}'] = (
        g.transform(lambda x: x.shift(TARGET_HORIZON).rolling(w, min_periods=2).std().fillna(0))
         .astype(np.float32)
    )

eps = 1e-6

df['trend_7_28'] = (
    df['roll_mean_7'] / (df['roll_mean_28'] + eps)
).clip(0, 10).astype(np.float32)

df['trend_7_56'] = (
    df['roll_mean_7'] / (df['roll_mean_56'] + eps)
).clip(0, 10).astype(np.float32)

df['volatility_28'] = (
    df['roll_std_28'] / (df['roll_mean_28'] + eps)
).clip(0, 10).astype(np.float32)

df['roll_diff_7_56'] = (
    df['roll_mean_7'] - df['roll_mean_56']
).astype(np.float32)

df['ItemCode_cat'] = df['ItemCode'].astype('category')

numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)

logger.info("Preparing model DataFrame …")

df['sku_first_date'] = df.groupby('ItemCode')['Date'].transform('min')
df['sku_first_date'] = df['sku_first_date'].fillna(DATA_START)
df['valid_lag'] = True  
df_model = df.copy()

df_model.drop(columns=['sku_first_date', 'valid_lag'], inplace=True, errors='ignore')

logger.info(f"df_model shape after fix: {df_model.shape}")

val_split_date = DATA_END - pd.Timedelta(days=TARGET_HORIZON)

train_df = df_model[df_model['Date'] <= val_split_date].copy()
val_df   = df_model[df_model['Date'] >  val_split_date].copy()

logger.info(f"Train: {train_df.shape}  |  Val: {val_df.shape}")

features = [
    'dow', 'month', 'week',
    'dow_sin', 'dow_cos',
    'month_sin', 'month_cos',
    'is_saturday',
    'is_quarter_end_month',
    'lag_56', 'lag_63', 'lag_70', 'lag_84',
    'roll_mean_7', 'roll_mean_14', 'roll_mean_28', 'roll_mean_56',
    'roll_std_7', 'roll_std_14', 'roll_std_28', 'roll_std_56',
    'trend_7_28', 'trend_7_56', 'volatility_28', 'roll_diff_7_56',
    'sku_mean', 'sku_std', 'sku_median', 'sku_max', 'sku_nonzero_rate',
    'is_holiday', 'days_to_next_holiday', 'days_since_last_holiday',
    'is_giotohungvuong', 'is_laodong', 'is_nhagiao', 'is_trungthu',
    'is_tet', 'days_to_tet', 'days_after_tet', 'pre_tet_window', 'post_tet_window',
    'is_pandemic_era',
    'ItemCode_cat',
]

target = 'Quantity'

sample_weight_train = train_df['profit_weight'].values
sample_weight_val   = val_df['profit_weight'].values

X_train = train_df[features]
y_train = train_df[target]
X_val   = val_df[features]
y_val   = val_df[target]

logger.info("Feature engineering complete.")
logger.info(f"Features: {len(features)}")
logger.info(f"X_train: {X_train.shape}  X_val: {X_val.shape}")
logger.info(f"SKUs predict-zero (WRMSSE safe): {len(predict_zero_skus)}")

gc.collect()

2026-05-20 12:12:16,985 INFO Loading raw transactions …
2026-05-20 12:12:19,095 INFO Computing profit weights …
2026-05-20 12:12:19,177 INFO Zero-weight SKUs (predict 0): 1816
2026-05-20 12:12:19,179 INFO Building net daily sales …
2026-05-20 12:12:19,428 INFO Never-sold SKUs (predict 0): 150
2026-05-20 12:12:19,430 INFO Total predict-zero SKUs: 1816
2026-05-20 12:12:19,433 INFO SKUs entering training pipeline: 14156
2026-05-20 12:12:19,470 INFO Building per-SKU date grid …
2026-05-20 12:12:26,583 INFO Grid shape: (12617454, 2)
2026-05-20 12:12:40,725 INFO Computing SKU static stats …
2026-05-20 12:12:44,223 INFO Computing lag features …
2026-05-20 12:12:45,512 INFO Computing rolling features …
2026-05-20 12:13:26,266 INFO Preparing model DataFrame …
2026-05-20 12:13:33,953 INFO df_model shape after fix: (12617454, 48)
2026-05-20 12:13:37,741 INFO Train: (11833815, 48)  |  Val: (783639, 48)
2026-05-20 12:13:38,594 INFO Feature engineering complete.
2026-05-20 12:13:38,595 INFO Features

0

## 6. Thiết Kế Tập Huấn Luyện & Mô Hình Hóa Với LightGBM
Phân chia tập huấn luyện và tập kiểm định nghiêm ngặt theo trục thời gian (Temporal Validation Split) để chống rò rỉ dữ liệu.

In [ ]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger(__name__)

TARGET_HORIZON = 56
SHIFT_DAY = 28
DATA_END = pd.Timestamp('2025-09-05')
VAL_START = DATA_END - pd.Timedelta(days=SHIFT_DAY)

train_daily_net = daily_net[daily_net['Date'] <= VAL_START]

naive_mse_list = []
for sku, grp in train_daily_net.groupby('ItemCode'):
    q = grp.sort_values('Date')['Quantity'].clip(lower=0).values
    naive_mse = float(np.mean((q[1:] - q[:-1]) ** 2)) if len(q) >= 2 else 0.0
    naive_mse_list.append({'ItemCode': sku, 'naive_mse': naive_mse})

naive_df = pd.DataFrame(naive_mse_list)

raw = pd.read_csv('data/raw/train.csv', dtype=str)
raw.columns = raw.columns.str.strip().str.replace(' ', '_')

def parse_vnd(s):
    return pd.to_numeric(
        s.astype(str).str.replace(',', '.', regex=False)
                     .str.replace(' ', '', regex=False),
        errors='coerce'
    )

raw['SalesAmount'] = parse_vnd(raw['SalesAmount'])
raw['Cost_Amount'] = parse_vnd(raw['Cost_Amount'])
raw['profit_line'] = raw['SalesAmount'] - raw['Cost_Amount']

sku_profit = raw.groupby('ItemCode')['profit_line'].sum().clip(lower=0)
sku_weight = (sku_profit / sku_profit.sum()).rename('profit_weight').reset_index()

weight_df = naive_df.merge(sku_weight, on='ItemCode', how='left')
weight_df['profit_weight'] = weight_df['profit_weight'].fillna(0)

weight_df['sample_weight'] = np.where(
    weight_df['naive_mse'] > 0,
    weight_df['profit_weight'] / weight_df['naive_mse'],
    0.0
)

sw_sum = weight_df['sample_weight'].sum() + 1e-9
weight_df['sample_weight'] = (
    weight_df['sample_weight'] / sw_sum * len(weight_df)
).astype(np.float32)

df_model = df_model.sort_values(['ItemCode', 'Date']).reset_index(drop=True)

train_mask = df_model['Date'] <= VAL_START
val_mask = df_model['Date'] > VAL_START

train_df = df_model[train_mask].copy()
val_df = df_model[val_mask].copy()

train_df = train_df.merge(weight_df[['ItemCode', 'sample_weight']], on='ItemCode', how='left')
train_df['sample_weight'] = train_df['sample_weight'].fillna(0).astype(np.float32)

val_df = val_df.merge(weight_df[['ItemCode', 'sample_weight']], on='ItemCode', how='left')
val_df['sample_weight'] = val_df['sample_weight'].fillna(0).astype(np.float32)

sku_stats_clustering = train_df.groupby('ItemCode')[target].agg(
    mean_qty='mean',
    std_qty='std',
    max_qty='max',
    nonzero_rate=lambda x: (x > 0).mean()
).reset_index()

sku_stats_clustering['cv'] = sku_stats_clustering['std_qty'] / (sku_stats_clustering['mean_qty'] + 1e-9)

conditions = [
    (sku_stats_clustering['nonzero_rate'] >= 0.7) & (sku_stats_clustering['cv'] <= 1.5),
    (sku_stats_clustering['cv'] > 2.0) & (sku_stats_clustering['max_qty'] >= 4 * sku_stats_clustering['mean_qty'])
]
choices = ['Stable', 'Spike']

sku_stats_clustering['demand_type'] = np.select(conditions, choices, default='Intermittent')

segment_configs = {
    'Stable': {
        'tweedie_variance_power': 1.10, 
        'num_leaves': 127,
        'learning_rate': 0.04,
        'min_data_in_leaf': 20,
        'colsample_bytree': 0.8 
    },
    'Spike': {
        'tweedie_variance_power': 1.30,
        'num_leaves': 63,
        'learning_rate': 0.03,
        'min_data_in_leaf': 50,
        'bagging_fraction': 0.7,
        'colsample_bytree': 0.7
    },
    'Intermittent': {
        'tweedie_variance_power': 1.85, 
        'num_leaves': 15, 
        'learning_rate': 0.015,
        'min_data_in_leaf': 150, 
        'bagging_fraction': 0.8
    }
}

val_df['y_pred'] = 0.0
models = {}

event_multiplier = np.where((train_df['is_holiday'] == 1) | (train_df['Date'].dt.dayofweek >= 5), 1.2, 1.0)

for seg_name, seg_params in segment_configs.items():
    logger.info(f"Training Segment: {seg_name}")
    
    seg_skus = sku_stats_clustering[sku_stats_clustering['demand_type'] == seg_name]['ItemCode']
    
    if len(seg_skus) == 0:
        continue

    X_train_seg = train_df[train_df['ItemCode'].isin(seg_skus)][features]
    y_train_seg = train_df[train_df['ItemCode'].isin(seg_skus)][target].astype(np.float32)
    
    w_train_seg = train_df[train_df['ItemCode'].isin(seg_skus)]['sample_weight'].values
    
    event_mult_seg = event_multiplier[train_df['ItemCode'].isin(seg_skus)]
    w_train_seg = w_train_seg * event_mult_seg

    X_val_seg = val_df[val_df['ItemCode'].isin(seg_skus)][features]
    y_val_seg = val_df[val_df['ItemCode'].isin(seg_skus)][target].astype(np.float32)
    w_val_seg = val_df[val_df['ItemCode'].isin(seg_skus)]['sample_weight'].values

    train_data = lgb.Dataset(X_train_seg, label=y_train_seg, weight=w_train_seg, categorical_feature=['ItemCode_cat'], free_raw_data=False)
    val_data = lgb.Dataset(X_val_seg, label=y_val_seg, weight=w_val_seg, reference=train_data, categorical_feature=['ItemCode_cat'], free_raw_data=False)

    base_params = {
        'objective': 'tweedie',
        'metric': 'tweedie',
        'max_depth': -1,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 1,
        'lambda_l1': 0.05,
        'lambda_l2': 0.1,
        'max_bin': 255,
        'seed': 42,
        'verbose': -1,
        'n_jobs': -1,
    }
    
    base_params.update(seg_params)

    model = lgb.train(
        base_params,
        train_data,
        num_boost_round=6000, 
        valid_sets=[val_data],
        valid_names=['val'],
        callbacks=[
            lgb.early_stopping(stopping_rounds=300, verbose=False),
        ],
    )

    models[seg_name] = model
    
    y_pred_seg = np.clip(model.predict(X_val_seg), 0, None)
    val_df.loc[val_df['ItemCode'].isin(seg_skus), 'y_pred'] = y_pred_seg

rmse = np.sqrt(mean_squared_error(val_df[target], val_df['y_pred']))
logger.info(f"Overall Val RMSE: {rmse:.4f}")

wrmsse_parts = []
for sku, grp in val_df.groupby('ItemCode'):
    row = weight_df[weight_df['ItemCode'] == sku]
    if row.empty:
        continue
    w_sku = float(row['profit_weight'].values[0])
    naive_mse = float(row['naive_mse'].values[0])
    if w_sku == 0 or naive_mse == 0:
        continue
    mse_sku = float(np.mean((grp[target].values - grp['y_pred'].values) ** 2))
    rmsse_sku = np.sqrt(mse_sku / naive_mse)
    wrmsse_parts.append(w_sku * rmsse_sku)

wrmsse_val = sum(wrmsse_parts)
logger.info(f"Overall Val WRMSSE: {wrmsse_val:.4f}")

gc.collect()

2026-05-20 12:15:59,918 INFO Training Segment: Stable
2026-05-20 12:16:01,214 INFO Training Segment: Spike
2026-05-20 12:19:57,004 INFO Training Segment: Intermittent
2026-05-20 12:20:01,857 INFO Overall Val RMSE: 2.6048
2026-05-20 12:20:16,779 INFO Overall Val WRMSSE: 0.3588


0

## 7. Khởi Tạo Dự Báo & Định Dạng File Nộp Bài
Suy luận lượng bán tương lai, áp dụng hàm chặn dưới bằng 0 và xoay ma trận sang định dạng hàng ngang khớp 100% cấu trúc yêu cầu của ban tổ chức.

In [ ]:
max_date = df_model['Date'].max()
test_data = df_model[df_model['Date'] > (max_date - pd.Timedelta(days=56))].copy()

test_data['preds'] = 0.0

for seg_name, model in models.items():
    seg_skus = sku_stats_clustering[sku_stats_clustering['demand_type'] == seg_name]['ItemCode']
    mask = test_data['ItemCode'].isin(seg_skus)
    if mask.sum() > 0:
        test_data.loc[mask, 'preds'] = np.clip(model.predict(test_data.loc[mask, features]), 0, None)

pivot_df = test_data.pivot_table(
    index='ItemCode',
    columns='Date',
    values='preds',
    aggfunc='mean'
).fillna(0)

pivot_df = pivot_df.sort_index(axis=1)

sundays = pivot_df.columns[pivot_df.columns.dayofweek == 6]
pivot_df[sundays] = 0.0

val_preds = pivot_df.iloc[:, :28].copy()
eval_preds = pivot_df.iloc[:, 28:56].copy()

val_preds.columns = [f'F{i}' for i in range(1, 29)]
eval_preds.columns = [f'F{i}' for i in range(1, 29)]

val_preds = val_preds.reset_index()
eval_preds = eval_preds.reset_index()

val_preds['id'] = val_preds['ItemCode'] + '_validation'
eval_preds['id'] = eval_preds['ItemCode'] + '_evaluation'

val_preds = val_preds.drop(columns=['ItemCode'])
eval_preds = eval_preds.drop(columns=['ItemCode'])

sub_model = pd.concat([val_preds, eval_preds], ignore_index=True)

cols = ['id'] + [f'F{i}' for i in range(1, 29)]
sub_model = sub_model[cols]

sample_sub = pd.read_csv('data/raw/sample_submission.csv')
final_sub = sample_sub[['id']].merge(sub_model, on='id', how='left').fillna(0)

final_sub.to_csv('submissions/submission.csv', index=False)